In [9]:
import tensorflow as tf
import numpy as np

In [10]:
class NeuralNetworkv1(tf.Module):
    def __init__(self, layers, learning_rate=0.01):
        super().__init__()
        self.layers = layers 
        
        self.weights_dense = [
            tf.Variable(tf.random.normal(shape=[layers[i], layers[i+1]]) * 0.1, name=f'W_dense_{i}') 
            for i in range(len(layers) - 1)
        ]
        
        self.optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate)

    def calculate_loss(self, y_true, y_pred):
        return tf.reduce_mean(tf.square(y_true - y_pred)) 

    def forward_pass(self, x):
        a = x
        for i in range(len(self.layers) - 1):
            z = tf.matmul(a, self.weights_dense[i])
            if i < len(self.layers) - 2:
                a = tf.nn.relu(z)
            else:
                a = z 
        return a

In [11]:
class SimpleCNN(NeuralNetworkv1):
    def __init__(self, input_shape, kernel_sizes, dense_layers, strides=1, padding='VALID'):
        # input_shape should be [batch, height, width, channels] for TF
        h, w = input_shape[1], input_shape[2]
        
        for k in kernel_sizes:
            if padding == 'VALID':
                h, w = h - k + 1, w - k + 1
        
        dense_layers.insert(0, h * w)
        super().__init__(dense_layers)
        
        self.kernel_sizes = kernel_sizes
        self.strides = strides
        self.padding = padding
        
        # In TF, kernels are shaped [filter_height, filter_width, in_channels, out_channels]
        # For simplicity, we assume 1 input channel and 1 output channel per layer
        self.kernels = [
            tf.Variable(tf.random.normal(mean=0, stddev=0.1, shape=[k, k, 1, 1], dtype=tf.float32), name=f'Conv_{k}') 
            for k in self.kernel_sizes
        ]
        
        self.c_loss_history = [] 

    def forward_pass_c(self, x):
        current_input = x
        
        for kernel in self.kernels:
            # tf.nn.conv2d replaces our slow Python loops!
            # It expects strides as a list: [batch_stride, height_stride, width_stride, channel_stride]
            z = tf.nn.conv2d(current_input, kernel, strides=[1, self.strides, self.strides, 1], padding=self.padding)
            current_input = tf.nn.relu(z)
            
        return current_input
    
    def train_cnn(self, x_train, y_train, epochs, learning_rate=0.01):
        # Update the optimizer learning rate
        self.optimizer.learning_rate.assign(learning_rate)
        
        for epoch in range(epochs):
            epoch_loss = 0
            
            for x, y in zip(x_train, y_train):
                # Reshape x to [batch=1, height, width, channels=1] for TF conv2d
                x_tensor = tf.cast(tf.reshape(x, [1, x.shape[0], x.shape[1], 1]), dtype=tf.float32)
                y_tensor = tf.cast(tf.reshape(y, [1, 1]), dtype=tf.float32)
                
                with tf.GradientTape() as tape:
                    # 1. Feature Extraction
                    z = self.forward_pass_c(x_tensor)
                    
                    # 2. Flatten (TF style)
                    flattened = tf.reshape(z, [1, -1])
                    
                    # 3. Classification
                    pred = self.forward_pass(flattened)
                    
                    # 4. Loss
                    loss = self.calculate_loss(y_tensor, pred)
                    
                # FIX: We must update BOTH the CNN kernels and the Dense weights!
                all_trainable_variables = self.kernels + self.weights_dense
                
                # The Tape calculates the chain rule for the entire network instantly
                gradients = tape.gradient(loss, all_trainable_variables)
                self.optimizer.apply_gradients(zip(gradients, all_trainable_variables))
                
                epoch_loss += loss.numpy()
                
            avg_loss = epoch_loss / len(x_train)
            self.c_loss_history.append(avg_loss)
            
            if (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch + 1} | Loss: {avg_loss:.4f}")

In [12]:
# ==========================================
# THE TEST: Ser Duncan vs Aerion Brightflame
# ==========================================
shield_dunk_1 = np.array([[1, 0, 0, 0, 0], [0, 1, 0, 0, 0], [0, 0, 1, 0, 0], [0, 0, 0, 1, 0], [0, 0, 0, 0, 1]])
shield_aerion_1 = np.array([[0, 0, 0, 0, 0], [0, 1, 1, 1, 0], [0, 1, 1, 1, 0], [0, 1, 1, 1, 0], [0, 0, 0, 0, 0]])

X_train = [shield_dunk_1, shield_aerion_1]
y_train = [1, 0]

# Initialize: 1 image, 5x5, 1 channel
model = SimpleCNN(input_shape=[1, 5, 5, 1], kernel_sizes=[3], dense_layers=[10, 1])
model.train_cnn(X_train, y_train, epochs=100, learning_rate=0.05)

Epoch 10 | Loss: 0.4989
Epoch 20 | Loss: 0.4933
Epoch 30 | Loss: 0.4652
Epoch 40 | Loss: 0.3077
Epoch 50 | Loss: 0.0219
Epoch 60 | Loss: 0.0009
Epoch 70 | Loss: 0.0004
Epoch 80 | Loss: 0.0002
Epoch 90 | Loss: 0.0001
Epoch 100 | Loss: 0.0000
